# Grupos de Participantes

Os dados são divididos em 5 grupos:

    1A – Pacientes com anotações, treinados pelo Poppy
    
    1B – Pacientes sem anotações, treinados pelo Poppy
    
    2A – Adultos saudáveis com anotações, treinados pelo Poppy
    
    2B – Adultos saudáveis sem anotações, treinados pelo Poppy
    
    3 – Participantes saudáveis simulando erros de execução (com anotações de tipo de erro)

(Poppy é um robozinho fisioterapeuta)

# Modalidades de Captura

Para cada grupo, os dados foram coletados por diferentes sistemas:

    1. Vicon (só no grupo 3) — captura de movimento profissional com 17 marcadores físicos no corpo, gravando posição 3D + orientação em quaternion de cada marcador (119 valores por frame).
    
    2. Kinect — esqueleto com 25 articulações, também com posição 3D + quaternion (175 valores por frame). Presente em todos os grupos.
    
    3. OpenPose — detecta 14 articulações a partir de vídeo, mas só em 2D (x, y). Usa o modelo COCO.
    
    4. BlazePose — detecta 33 pontos-chave em 3D (x, y, z), onde z é a profundidade em relação ao plano do vídeo.

Além disso, há vídeos e anotações para os grupos que possuem rótulos.

MediaPipe usa BlazePose

# Exercícios

Os exercícios do dataset foram definidos em conjunto com fisioterapeutas, focados no alongamento da coluna para o tratamento de dor lombar (low back pain). As siglas encontradas nos nomes dos arquivos representam as três categorias de movimento:

- CTK (Hiding Face / Exercício de Respiração): O paciente flexiona os membros superiores a 90 graus no ombro e no cotovelo (como se estivesse "escondendo o rosto") combinando o movimento com o controle respiratório.

- ELK (Flank Stretch / Flexão Lateral): Consiste na inclinação lateral do tronco, alternando para o lado esquerdo e para o lado direito.

- RTK (Torso Rotation / Rotação de Tronco): Um movimento focado na rotação do tronco sobre o próprio eixo, girando para a esquerda e depois para a direita.

In [8]:
import pandas as pd
import numpy as np
import os
import json
import pandas as pd
import matplotlib.pyplot as plt
import xml.etree.ElementTree as ET
import glob

# Carregar coordenadas do BlazePose

In [9]:
def load_blazepose(json_path):
    with open(json_path, 'r') as f:
        data = json.load(f)
    frames_dict = data.get('positions', data)
    rows = []
    
    for frame_key, joints in frames_dict.items():
        try:
            row = {'frame': int(float(frame_key))}
            if isinstance(joints, dict):
                for joint_name, coords in joints.items():
                    if isinstance(coords, list) and len(coords) >= 3:
                        row[f"{joint_name}_x"] = coords[0]
                        row[f"{joint_name}_y"] = coords[1]
                        row[f"{joint_name}_z"] = coords[2]
            if len(row) > 1:
                rows.append(row)
        except (ValueError, TypeError):
            continue
            
    df = pd.DataFrame(rows)
    if 'frame' in df.columns:
        df = df.sort_values('frame').reset_index(drop=True)
    return df

# Carregar rótulos do Anvil

In [ ]:
def load_anvil(anvil_path):
    if not os.path.exists(anvil_path):
        return pd.DataFrame()
        
    tree = ET.parse(anvil_path)
    root = tree.getroot()
    anotacoes = []
    
    # Função interna para proteger a conversão de números
    def safe_float(val):
        if not val:
            return 0.0
        try:
            return float(val)
        except ValueError:
            return 0.0
    
    for track in root.findall('.//track'):
        track_name = track.get('name')
        for el in track.findall('.//el'):
            start = safe_float(el.get('start'))
            end = safe_float(el.get('end'))
            
            label = el.get('type') or el.get('label') or ""
            for attr in el.findall('.//attribute'):
                if attr.text:
                    label += f" {attr.text}"
            
            label = label.strip()
            if not label:
                label = "Unknown"
                
            anotacoes.append({
                'track': track_name,
                'start_frame': start,
                'end_frame': end,
                'label': label
            })
            
    return pd.DataFrame(anotacoes)

# Mesclar dados

In [11]:
def merge_data_multi(df_pose, df_labels, suffix="_ann1"):
    eval_col = f'evaluation{suffix}'
    err_col = f'error_type{suffix}'
    
    # Valores padrão
    df_pose[eval_col] = 'Correct' 
    df_pose[err_col] = 'None'
    
    if df_labels.empty:
        df_pose[eval_col] = 'Unknown'
        df_pose[err_col] = 'Unknown'
        return df_pose
        
    for _, row in df_labels.iterrows():
        mask = (df_pose['frame'] >= row['start_frame']) & (df_pose['frame'] <= row['end_frame'])
        if row['track'] == 'Global evaluation':
            df_pose.loc[mask, eval_col] = row['label']
        elif row['track'] == 'Global error':
            df_pose.loc[mask, err_col] = row['label']
            
    return df_pose

# Contruindo Dataset


In [ ]:
def build_dataset_full(base_dir, group_name="group1A"):
    blazepose_dir = os.path.join(base_dir, group_name, "blazepose")
    all_recordings = []
    
    # Busca todos os JSONs na pasta do grupo
    json_files = glob.glob(os.path.join(blazepose_dir, "*.json"))
    
    for json_path in json_files:
        filename = os.path.basename(json_path)
        print(f"Processando: {filename}...")
        
        # Nome base para buscar os arquivos de anotação (remove o -BP-)
        base_name = filename.replace("-BP-", "-").replace(".json", "")
        
        # Extrai as coordenadas
        df_pose = load_blazepose(json_path)
        if df_pose.empty:
            continue
            
        # Busca recursivamente qualquer arquivo .anvil que contenha o nome base do vídeo
        search_pattern = os.path.join(base_dir, group_name, "**", f"*{base_name}*.anvil")
        anvil_files = glob.glob(search_pattern, recursive=True)
        
        # Ordena para garantir que ann1 e ann2 sejam consistentes 
        anvil_files.sort()
        
        # Se não encontrar nenhum anotador (como nos grupos 1B e 2B)
        if not anvil_files:
             df_pose = merge_data_multi(df_pose, pd.DataFrame(), suffix="_ann1")
        else:
            # Para cada arquivo de anotação encontrado, cria as colunas respectivas
            for idx, anvil_path in enumerate(anvil_files, start=1):
                df_labels = load_anvil(anvil_path)
                df_pose = merge_data_multi(df_pose, df_labels, suffix=f"_ann{idx}")
        
        # Adiciona os metadados
        parts = filename.replace(".json", "").split('-')
        if len(parts) >= 5:
            df_pose['group_id'] = parts[0]
            df_pose['exercise'] = parts[2]
            df_pose['recording_id'] = filename
            
        all_recordings.append(df_pose)
        
    return pd.concat(all_recordings, ignore_index=True) if all_recordings else pd.DataFrame()


diretorio_raiz = "keraal_sample_2022" 
grupos = ['group1A', 'group2A'] 
todos_os_dados = []

for grupo in grupos:
    print(f"\n--- Extraindo {grupo} ---")
    df_grupo = build_dataset_full(diretorio_raiz, group_name=grupo)
    todos_os_dados.append(df_grupo)

df_final = pd.concat(todos_os_dados, ignore_index=True)
df_final.to_csv("dataset_completo_multi_annotators.csv", index=False)
print(f"\n✅ Extração concluída! Dataset final tem {len(df_final)} linhas.")


--- Extraindo group1A ---
Processando: G1A-BP-CTK-R2-Brest-008.json...
Processando: G1A-BP-ELK-R5-Roscoff-024.json...
Processando: G1A-BP-ELK-R2-Brest-008.json...
Processando: G1A-BP-RTK-R4-Roscoff-031.json...
Processando: G1A-BP-RTK-R3-Brest-066.json...
Processando: G1A-BP-CTK-R2-Brest-032.json...
Processando: G1A-BP-RTK-R6-Roscoff-001.json...
Processando: G1A-BP-CTK-R3-Brest-083.json...
Processando: G1A-BP-RTK-R2-Brest-015.json...
Processando: G1A-BP-CTK-R4-Roscoff-053.json...
Processando: G1A-BP-ELK-R1-Brest-052.json...
Processando: G1A-BP-RTK-R6-Roscoff-027.json...
Processando: G1A-BP-CTK-R5-Roscoff-058.json...
Processando: G1A-BP-RTK-R5-Roscoff-053.json...
Processando: G1A-BP-CTK-R2-Brest-014.json...
Processando: G1A-BP-ELK-R1-Brest-075.json...
Processando: G1A-BP-RTK-R3-Brest-036.json...
Processando: G1A-BP-RTK-R2-Brest-007.json...
Processando: G1A-BP-ELK-R6-Roscoff-043.json...
Processando: G1A-BP-RTK-R2-Brest-047.json...
Processando: G1A-BP-RTK-R1-Brest-006.json...
Processando:

In [ ]:
def resolve_annotator_consensus(df):
    df_cons = df.copy()
    
    if 'evaluation_ann1' not in df_cons.columns or 'evaluation_ann2' not in df_cons.columns:
        return df_cons

    def get_eval_consensus(row):
        e1, e2 = row['evaluation_ann1'], row['evaluation_ann2']
        if e1 == e2: return e1
        elif 'Incorrect' in [e1, e2]: return 'Incorrect'
        elif e1 == 'Unknown': return e2
        elif e2 == 'Unknown': return e1
        else: return 'Disagreement'

    def get_err_consensus(row):
        err1, err2 = row['error_type_ann1'], row['error_type_ann2']
        if err1 == err2: return err1
        elif err1 in ['None', 'Unknown']: return err2
        elif err2 in ['None', 'Unknown']: return err1
        else: return f"{err1} | {err2}"

    df_cons['evaluation'] = df_cons.apply(get_eval_consensus, axis=1)
    df_cons['error_type'] = df_cons.apply(get_err_consensus, axis=1)
    
    colunas_para_remover = ['evaluation_ann1', 'evaluation_ann2', 'error_type_ann1', 'error_type_ann2']
  
    df_cons = df_cons.drop(columns=colunas_para_remover, errors='ignore')

    return df_cons

df_final = resolve_annotator_consensus(df_final)

In [14]:
df_final.head(100).to_csv("amostra_visualizacao.csv", index=False)

In [15]:
df_final.head()

,frame,Nose_x,Nose_y,Nose_z,Left_eye_inner_x,Left_eye_inner_y,Left_eye_inner_z,Left_eye_x,Left_eye_y,Left_eye_z,...,Left_foot_index_y,Left_foot_index_z,Right_foot_index_x,Right_foot_index_y,Right_foot_index_z,group_id,exercise,recording_id,evaluation,error_type
0,1,0.465428,0.508279,-0.364005,0.468834,0.498485,-0.351307,0.470634,0.498326,-0.351301,...,0.955784,-0.089269,0.442028,0.961440,-0.048266,G1A,CTK,G1A-BP-CTK-R2-Brest-008.json,Incorrect,SmallError Error3 BothArms
1,2,0.465258,0.507820,-0.315705,0.468813,0.498175,-0.303334,0.470632,0.498009,-0.303334,...,0.955928,-0.089271,0.440529,0.961924,-0.059211,G1A,CTK,G1A-BP-CTK-R2-Brest-008.json,Incorrect,SmallError Error3 BothArms
2,3,0.464887,0.507599,-0.304219,0.468693,0.498019,-0.291850,0.470576,0.497829,-0.291850,...,0.956043,-0.103877,0.439731,0.962260,-0.078299,G1A,CTK,G1A-BP-CTK-R2-Brest-008.json,Incorrect,SmallError Error3 BothArms
3,4,0.464533,0.507498,-0.293234,0.468541,0.497924,-0.280643,0.470489,0.497716,-0.280644,...,0.956066,-0.115636,0.439253,0.962492,-0.089009,G1A,CTK,G1A-BP-CTK-R2-Brest-008.json,Incorrect,SmallError Error3 BothArms
4,5,0.464435,0.507466,-0.293860,0.468503,0.497888,-0.281064,0.470475,0.497669,-0.281063,...,0.956075,-0.126131,0.438931,0.962627,-0.098904,G1A,CTK,G1A-BP-CTK-R2-Brest-008.json,Incorrect,SmallError Error3 BothArms
